# **Exploring BioPAX Reactome *Homo sapiens* File**

- BioPax: Biological Pathway Exchange

# Import Libraries and Configurations

In [1]:
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from collections import defaultdict
from pybiopax import model_from_owl_file
from pybiopax.biopax import (
    Complex,
    Pathway,
    Protein,
)

# Dictionary to index pathway objects by Reactome ID
PATHWAYS = dict()

HSA_FILE = 'Homo-sapiens_v96.owl'
HSA_FILE_PATH = f'../data/external/reactome/{HSA_FILE}'

# Dynamic programming caches
_PROTEIN_COLLECTION_CACHE = dict()
_INTERACTION_CACHE = dict()

# Functions

## BioPAX Pathway DAG

In [2]:
def collect_proteins(entity):
    """
    Recursively collect all protein objects contained within a physical entity.

    This function traverses a physical entity hierarchy and returns all
    :class:`Protein` instances contained directly or indirectly within the
    input entity. It supports individual proteins, protein entity sets
    (via ``member_physical_entity``), and :class:`Complex` objects.
    Results are memoized using a global cache to avoid redundant traversals
    of previously visited entities.

    Parameters
    ----------
    entity : Protein or Complex or None
        The physical entity from which to collect proteins. The entity may be
        a single :class:`Protein`, a protein entity set, a
        :class:`Complex`, or ``None``.

    Returns
    -------
    set of Protein
        A set containing all unique :class:`Protein` instances found within
        the entity. Returns an empty set if ``entity`` is ``None``.

    Notes
    -----
    Protein entity sets are recursively expanded through their
    ``member_physical_entity`` attribute, whereas complexes are traversed
    through their ``component`` attribute. Cache entries are keyed by the
    entity's ``uid`` attribute when available; otherwise, Python's built-in
    :func:`id` is used. The cache is stored in the global
    ``_PROTEIN_COLLECTION_CACHE`` dictionary.

    Examples
    --------
    >>> proteins = collect_proteins(complex_entity)
    >>> len(proteins)
    5
    """
    # None object
    if entity is None:
        return set()

    # Get entity object ID
    entity_id = getattr(entity, 'uid', id(entity))

    # Check cache
    if entity_id in _PROTEIN_COLLECTION_CACHE:
        return _PROTEIN_COLLECTION_CACHE[entity_id]

    # Create a set to store Protein objects
    proteins = set()

    # Get EntitySet members, if present
    member_entities = getattr(
        entity, 'member_physical_entity', None
    )

    # Protein object
    if isinstance(entity, Protein):
        if member_entities:
            for member in member_entities:
                proteins.update(
                    collect_proteins(member)
                )
        else:
            proteins.add(entity)

    # Complex object
    if isinstance(entity, Complex):
        for component in entity.component:
            proteins.update(
                collect_proteins(component)
            )

    # Cache result
    _PROTEIN_COLLECTION_CACHE[entity_id] = proteins

    return proteins

In [3]:
def get_interaction_proteins(interaction):
    """
    Extract all proteins participating in an interaction.

    This function collects all unique :class:`Protein` instances associated
    with an interaction by traversing its ``controller``, ``left``, and
    ``right`` participants. Protein extraction is performed recursively using
    :func:`collect_proteins`, allowing proteins nested within complexes or
    entity sets to be included. Results are memoized to avoid repeated
    traversal of previously processed interactions.

    Parameters
    ----------
    interaction : object
        An interaction object exposing one or more of the attributes
        ``controller``, ``left``, and ``right``. Each attribute is expected
        to contain an iterable of physical entities.

    Returns
    -------
    set of Protein
        A set containing all unique proteins participating in the interaction.

    Notes
    -----
    Cache entries are keyed by the interaction's ``uid`` attribute when
    available; otherwise, Python's built-in :func:`id` is used. The cache is
    stored in the global ``_INTERACTION_CACHE`` dictionary.

    Examples
    --------
    >>> proteins = get_interaction_proteins(interaction)
    >>> len(proteins)
    12
    """
    # Get interaction object ID
    interaction_id = getattr(
        interaction, 'uid', id(interaction)
    )

    # Check if the interaction has been visited previously
    if interaction_id in _INTERACTION_CACHE:
        return _INTERACTION_CACHE[interaction_id]

    # Create a set to store Protein objects
    proteins = set()

    # Iterate over the attributes of interest
    for attr in ('controller', 'left', 'right'):
        # Get the object(s) contained in the attribute
        entities = getattr(interaction, attr, None)

        # Attribute with an empty object
        if not entities:
            continue
        
        # Iterate over entity(ies) to extract its proteins
        for entity in entities:
            proteins.update(
                collect_proteins(entity)
            )

    # Add interaction and its proteins to the cache
    _INTERACTION_CACHE[interaction_id] = proteins

    return proteins

In [4]:
def build_pathway_dag(G, pathway, parent_id, visited=None):
    """
    Build a directed acyclic graph (DAG) representation of a pathway hierarchy.

    This function recursively traverses a pathway and populates a directed
    graph with pathway steps, interactions, nested pathways, and their
    associated proteins. The resulting graph captures the hierarchical
    structure of the pathway and the relationships between pathway steps,
    biological processes, and participating proteins.

    Nodes are annotated with a human-readable name and a ``type``
    attribute indicating their biological type (e.g., ``PathwayStep``,
    ``Protein``, or the interaction class name). Edges represent containment
    or participation relationships.

    Parameters
    ----------
    G : networkx.DiGraph
        Directed graph to populate. The graph is modified in place.
    pathway : Pathway
        Root pathway to traverse.
    parent_id : str
        Identifier of the parent node to which pathway steps will be
        connected.
    visited : set, optional
        Set of pathway identifiers that have already been traversed. This is
        used internally to prevent infinite recursion when pathways reference
        one another. If ``None``, an empty set is created.

    Returns
    -------
    networkx.DiGraph
        The populated directed graph.

    Notes
    -----
    The traversal proceeds as follows:

    1. Add pathway steps as child nodes of ``parent_id``.
    2. Add each process associated with a pathway step.
    3. Recursively traverse nested pathways.
    4. For interaction processes, add all participating proteins obtained via
       :func:`get_interaction_proteins`.

    Existing nodes are reused rather than recreated, allowing multiple
    relationships to converge on the same biological entity.

    Examples
    --------
    >>> import networkx as nx
    >>> G = nx.DiGraph()
    >>> build_pathway_dag(
    ...     G,
    ...     pathway=root_pathway,
    ...     parent_id=root_pathway.uid
    ... )
    >>> G.number_of_nodes()
    153
    """
    # Check if the set of visited pathways has been initialized
    if visited is None:
        visited = set()

    # Get pathway object ID
    pathway_id = getattr(pathway, 'uid', id(pathway))

    # Check if the pathway has already been visited
    if pathway_id in visited:
        return G

    # Add the pathway to the set of visited pathways
    visited.add(pathway_id)

    # Get the order of the pathway components
    pathway_order = getattr(pathway, 'pathway_order', None)
    
    if not pathway_order:
        return G

    # Iterate over the pathway step(s)
    for step in pathway_order:
        # Add pathway step node to the graph, if necessary
        if step.uid not in G:
            G.add_node(
                step.uid,
                name=getattr(
                    step, 'display_name', step.uid
                ),
                type='PathwayStep'
            )

        # Add edge from pathway to pathway step
        G.add_edge(parent_id, step.uid)

        # Iterate over the process(es) in the pathway step
        for process in step.step_process:
            # Add process node to the graph, if necessary
            if process.uid not in G:
                process_name = (
                    getattr(process, 'display_name', None)
                    or process.uid
                )

                if process_name == process.uid:
                    process_name = (
                        getattr(process, 'control_type', None)
                        or process.uid
                    )
                
                G.add_node(
                    process.uid,
                    name=process_name,
                    type=type(
                        process
                    ).__name__
                )

            # Add edge from pathway step to process
            G.add_edge(step.uid, process.uid)

            # Nested Pathway object
            if isinstance(process, Pathway):
                # Traverse the subpathway
                build_pathway_dag(
                    G=G,
                    pathway=process,
                    parent_id=process.uid,
                    visited=visited
                )
                continue

            # Get proteins related to the Interaction object
            proteins = get_interaction_proteins(process)

            # Iterate over the proteins
            for protein in proteins:
                # Add protein node to the graph, if necessary
                if protein.uid not in G:
                    protein_name = (
                        getattr(protein, 'display_name', None)
                        or protein.uid
                    )

                    G.add_node(
                        protein.uid,
                        name=protein_name,
                        type='Protein'
                    )

                # Add edge from process to protein
                G.add_edge(process.uid, protein.uid)

    return G

## Tripartite PSP Network

In [5]:
def project_pathway_steps(G):
    """
    Remove pathway step nodes from a pathway graph while preserving connectivity.

    This function projects a pathway graph onto its non-step nodes by removing
    all nodes whose ``type`` is ``"PathwayStep"``. Before each pathway
    step is removed, directed edges are added from each predecessor of the step
    to each of its successors, preserving reachability in the resulting graph.

    The input graph is copied before modification, leaving the original graph
    unchanged.

    Parameters
    ----------
    G : networkx.DiGraph
        Directed pathway graph containing nodes annotated with a
        ``type`` attribute.

    Returns
    -------
    networkx.DiGraph
        A copy of the input graph with all ``PathwayStep`` nodes removed and
        replacement edges added to preserve connectivity.

    Notes
    -----
    For each pathway step node, the function creates the Cartesian product of
    its predecessors and successors, adding an edge for every predecessor–
    successor pair before removing the step node.

    Examples
    --------
    >>> G_projected = project_pathway_steps(G)
    >>> any(
    ...     attr['type'] == 'PathwayStep'
    ...     for _, attr in G_projected.nodes(data=True)
    ... )
    False
    """
    # Create a copy of the graph
    G_proj = G.copy()

    # Get the pathway step nodes
    step_nodes = [
        n for n, attr in G_proj.nodes(data=True)
        if attr['type'] == 'PathwayStep'
    ]

    # Iterate over each pathway step node
    for step in step_nodes:
        # Get node parent(s) and child(ren)
        parents = tuple(G_proj.predecessors(step))
        children = tuple(G_proj.successors(step))

        # Add edges between parent(s) and child(ren)
        G_proj.add_edges_from(
            (p, c)
            for p in parents
            for c in children
        )

        # Remove the node from the graph
        G_proj.remove_node(step)

    return G_proj

In [6]:
def identify_pp_paths(G, source):
    """
    Enumerate all pathway-to-protein (PP) paths from a source node.

    This function traverses a directed acyclic graph (DAG) in topological
    order and yields every directed path from the specified source node to
    each reachable protein node. Protein nodes are identified by their
    ``type`` attribute equal to ``"Protein"``.

    Paths are propagated incrementally through the graph, allowing every
    distinct route from the source to a protein to be enumerated without
    performing repeated graph traversals.

    Parameters
    ----------
    G : networkx.DiGraph
        Directed acyclic graph containing nodes annotated with a
        ``type`` attribute.
    source : hashable
        Identifier of the source node from which pathway-to-protein paths are
        generated.

    Yields
    ------
    list
        A list of node identifiers representing a directed pathway-to-protein
        (PP) path from ``source`` to a protein node. Each yielded path begins
        with ``source`` and ends with a node whose ``type`` is
        ``"Protein"``.

    Notes
    -----
    This function assumes that ``G`` is a directed acyclic graph (DAG). The
    traversal relies on :func:`networkx.topological_sort` and propagates
    partial paths through successor nodes until protein leaves are reached.

    Examples
    --------
    >>> for path in identify_pp_paths(G, source='R-HSA-12345'):
    ...     print(path)
    ['R-HSA-12345', 'reaction1', 'P12345']
    ['R-HSA-12345', 'reaction2', 'complex1', 'P67890']
    """
    # Cache useful objects
    type = nx.get_node_attributes(G, 'type')
    successors = G.successors

    # Create a dictionary of empty lists to store paths
    paths = defaultdict(list)

    # Initialize the list of paths from the source node
    paths[source].append([source])

    # Iterate over the nodes in topological order
    for node in nx.topological_sort(G):
        # Get all paths that lead to the node
        current_paths = paths.get(node)

        # Skip unreachable node
        if not current_paths:
            continue

        # Output protein (leaf) paths
        if type[node] == 'Protein':
            yield from current_paths
            continue

        # Iterate over the node's child(ren)
        for child in successors(node):
            # Cache the list of paths from the child node
            child_paths = paths[child]

            # Iterate over the paths the lead to the node
            for path in current_paths:
                # Propagate paths to child
                child_paths.append(path + [child])

In [7]:
def export_network_tables(G):
    """
    Export a NetworkX graph as edge and node tables.

    This function converts a directed graph into two pandas DataFrames: one
    containing the graph edges and another containing the node attributes.

    Parameters
    ----------
    G : networkx.DiGraph
        Graph to export.

    Returns
    -------
    tuple of pandas.DataFrame
        A tuple ``(edges_df, nodes_df)`` where:

        - ``edges_df`` contains one row per directed edge with the default
        ``source`` and ``target`` columns produced by
        :func:`networkx.to_pandas_edgelist`.
        - ``nodes_df`` contains one row per node with an ``id`` column
        followed by all node attributes stored in the graph.

    Notes
    -----
    The edge table is generated using
    :func:`networkx.to_pandas_edgelist`, preserving the default
    ``source`` and ``target`` column names. The node table is
    constructed from the graph nodes and their associated attributes.

    Examples
    --------
    >>> edges_df, nodes_df = export_network_tables(G)
    >>> edges_df.head()
    >>> nodes_df.head()
    """
    # Create a DataFrame for the edge list
    edges_df = nx.to_pandas_edgelist(G)

    # Create a DataFrame for the node attributes
    nodes_df = pd.DataFrame(
        (
            {'id': n, **attrs}
            for n, attrs in G.nodes(data=True)
        )
    )

    return edges_df, nodes_df

In [8]:
def create_psp_network(pathway_id):
    """
    Create a pathway-semantics-protein (PSP) network from a pathway.

    This function constructs a simplified tripartite representation of a
    biological pathway by connecting the pathway to semantic descriptions of
    pathway-to-protein paths and, subsequently, to the associated protein nodes.
    The resulting network contains three node types:

    - ``Pathway``: the root pathway.
    - ``Semantics``: textual representations of the biological process sequence
    connecting a pathway to a protein.
    - ``Messenger RNA``: protein nodes identified from pathway interactions.

    The PSP network is generated through the following steps:

    1. Retrieve the pathway object from the global ``PATHWAYS`` collection.
    2. Construct the complete pathway DAG using :func:`build_pathway_dag`.
    3. Remove intermediate ``PathwayStep`` nodes using
    :func:`project_pathway_steps`.
    4. Enumerate all pathway-to-protein (PP) paths using
    :func:`identify_pp_paths`.
    5. Convert each PP path into a semantic representation by joining
    intermediate process names.
    6. Construct a tripartite network containing
    ``Pathway → Semantics → Messenger RNA`` relationships.
    7. Export the resulting graph as node and edge tables.

    Parameters
    ----------
    pathway_id : str
        Identifier of the pathway stored in the global ``PATHWAYS`` collection.

    Returns
    -------
    tuple of pandas.DataFrame
        A tuple ``(edges_df, nodes_df)`` containing the edge and node tables
        representing the PSP network, as produced by
        :func:`export_network_tables`.

    Raises
    ------
    KeyError
        If ``pathway_id`` is not present in the global ``PATHWAYS`` collection.

    Notes
    -----
    Semantic nodes are generated by concatenating the names of intermediate
    nodes along pathway-to-protein paths using ``" > "`` as a separator.
    Different PP paths that produce identical semantic strings are represented
    by the same semantic node.

    The ``Messenger RNA`` node type corresponds to proteins collected from the
    pathway graph. The naming follows the PSP network convention used for
    downstream analysis.

    Examples
    --------
    >>> edges_df, nodes_df = create_psp_network('R-HSA-123456')
    >>> nodes_df['type'].value_counts()
    """
    # Get the pathway object
    if pathway_id not in PATHWAYS:
        raise KeyError(
            f'Pathway ID {pathway_id!r} not found in PATHWAYS.'
        )

    pathway_obj = PATHWAYS[pathway_id]
    
    # Create an object to represent the pathway DAG
    G = nx.DiGraph()

    # Add the pathway node (root) to the graph
    G.add_node(
        pathway_obj.uid,
        name=pathway_obj.display_name,
        type='Pathway',
    )

    # Traverse the pathway
    build_pathway_dag(
        G=G,
        pathway=pathway_obj,
        parent_id=pathway_obj.uid,
    )

    # Project the pathway step nodes
    G_proj = project_pathway_steps(G)

    # Create an object to represent the PSP graph
    T = nx.DiGraph()

    # Add the pathway node to the tripartite graph
    nodes = G_proj.nodes
    pathway_name = nodes[pathway_obj.uid]['name']
    T.add_node(
        pathway_id,
        label=pathway_name,
        type='Pathway',
    )

    # Initialize semantics cache
    semantics_cache = dict()

    # Iterate over each pathway-protein (PP) path
    for path in identify_pp_paths(
        G=G_proj, source=pathway_obj.uid,
    ):
        # Get protein name
        protein_name = nodes[path[-1]]['name']

        # Add protein node to the graph, if necessary
        if protein_name not in T:
            T.add_node(
                protein_name,
                label=protein_name,
                type='Messenger RNA'
            )
        
        # Create PP path textual representation
        semantics = ' > '.join(
            nodes[n]['name']
            for n in path[1:-1]
        )

        # Retrieve or create semantics ID
        if semantics not in semantics_cache:
            # Create semantics ID
            semantics_id = (
                f'S{len(semantics_cache) + 1:05d}'
            )

            # Cache semantics ID
            semantics_cache[semantics] = semantics_id

            # Add semantics node to the graph
            T.add_node(
                semantics_id,
                label=semantics,
                type='Semantics',
            )
        else:
            # Retrieve the semantics ID
            semantics_id = semantics_cache[semantics]

        # Add edge from pathway to semantics
        T.add_edge(pathway_id, semantics_id)

        # Add edge from semantics to protein
        T.add_edge(semantics_id, protein_name)

    return export_network_tables(T)

# Exploration

## Create the Model

In [9]:
# Create a BioPAX model from the OWL file
model = model_from_owl_file(HSA_FILE_PATH)

Processing OWL elements:   0%|          | 0.00/545k [00:00<?, ?it/s]

## Index the Pathways

In [10]:
# Iterate over the Pathway objects
for pathway in model.get_objects_by_type(Pathway):
    # Iterate over the xrefs of the Pathway object
    for xref in pathway.xref:
        # Add the Reactome Pathway object to the dictionary
        if getattr(xref, 'db', None) == 'Reactome':
            reactome_id = xref.id
            PATHWAYS[reactome_id] = pathway
            break

## Signaling by WNT Pathway

In [11]:
pathway_id = 'R-HSA-195721'

# Create the tripartite pathway-semantics-protein network
edges_df, nodes_df = create_psp_network(pathway_id)

In [12]:
edges_df

,source,target
0,R-HSA-195721,S00001
1,R-HSA-195721,S00002
2,R-HSA-195721,S00003
3,R-HSA-195721,S00004
4,R-HSA-195721,S00005
...,...,...
1956,S00168,TCF7L2
1957,S00168,TCF7
1958,S00168,CTNNB1
1959,S00168,TCF3


In [13]:
nodes_df

,id,label,type
0,R-HSA-195721,Signaling by WNT,Pathway
1,USP34,USP34,Messenger RNA
2,S00001,TCF dependent signaling in response to WNT > A...,Semantics
3,APC,APC,Messenger RNA
4,S00002,Degradation of beta-catenin by the destruction...,Semantics
...,...,...,...
538,S00167,TCF dependent signaling in response to WNT > F...,Semantics
539,S00168,TCF dependent signaling in response to WNT > F...,Semantics
540,TCF7,TCF7,Messenger RNA
541,TCF3,TCF3,Messenger RNA


In [14]:
nodes_df \
    .query('type == "Semantics"')\
    .sort_values(by='label') \
    ['label'].to_list()

['Beta-catenin independent WNT signaling > Ca2+ pathway > ACTIVATION',
 'Beta-catenin independent WNT signaling > Ca2+ pathway > Activation of Calcineurin',
 'Beta-catenin independent WNT signaling > Ca2+ pathway > Activation of MAP3K7 in response to WNT',
 'Beta-catenin independent WNT signaling > Ca2+ pathway > Activation of NLK',
 'Beta-catenin independent WNT signaling > Ca2+ pathway > Active calmodulin binds CAMK2',
 'Beta-catenin independent WNT signaling > Ca2+ pathway > Autophosphorylation and activation of CAMK2',
 'Beta-catenin independent WNT signaling > Ca2+ pathway > CAMK2 binds MAP3K7',
 'Beta-catenin independent WNT signaling > Ca2+ pathway > Calcineurin binds and dephosphorylates NFAT1 in response to WNT/Ca2+ signaling',
 'Beta-catenin independent WNT signaling > Ca2+ pathway > Dissociation of CaM and CAMK2 autophosphorylation',
 'Beta-catenin independent WNT signaling > Ca2+ pathway > FZD recruits trimeric G-proteins',
 'Beta-catenin independent WNT signaling > Ca2+ pa

## Signaling by Receptor Tyrosine Kinases


In [15]:
pathway_id = 'R-HSA-9006934'

# Create the tripartite pathway-semantics-protein network
edges_df, nodes_df = create_psp_network(pathway_id)

In [16]:
edges_df

,source,target
0,R-HSA-9006934,S00001
1,R-HSA-9006934,S00002
2,R-HSA-9006934,S00003
3,R-HSA-9006934,S00004
4,R-HSA-9006934,S00005
...,...,...
4989,S00888,p-S295-PDE3B
4990,S00889,PKB beta
4991,S00889,p-S295-PDE3B
4992,S00890,PKB beta


In [17]:
nodes_df

,id,label,type
0,R-HSA-9006934,Signaling by Receptor Tyrosine Kinases,Pathway
1,DLG4,DLG4,Messenger RNA
2,S00001,Signaling by ERBB4 > ERBB4 binds DLG4,Semantics
3,RON_HUMAN,RON_HUMAN,Messenger RNA
4,S00002,Signaling by MST1 > MST1R autophosphorylates,Semantics
...,...,...,...
1504,S00888,Signaling by Type 1 Insulin-like Growth Factor...,Semantics
1505,PKB beta,PKB beta,Messenger RNA
1506,S00889,Signaling by Insulin receptor > Insulin recept...,Semantics
1507,S00890,Signaling by Type 1 Insulin-like Growth Factor...,Semantics


In [18]:
nodes_df \
    .query('type == "Semantics"')\
    .sort_values(by='label') \
    ['label'].to_list()

['Signaling by ALK > ACTIVATION',
 'Signaling by ALK > ALK binds ALKAL ligand',
 'Signaling by ALK > ALK gene expression',
 'Signaling by ALK > ALK-stimulated MYCN gene expression',
 'Signaling by ALK > Active ALK binds FRS2',
 'Signaling by ALK > Active ALK binds JAK3',
 'Signaling by ALK > Active ALK binds PLCG1',
 'Signaling by ALK > Active ALK binds SHC1',
 'Signaling by ALK > Active ALK binds SRC',
 'Signaling by ALK > Active ALK dimer binds IRS1',
 'Signaling by ALK > Active ALK phosphorylates FRS2',
 'Signaling by ALK > Active ALK phosphorylates IRS1',
 'Signaling by ALK > Active ALK phosphorylates JAK3',
 'Signaling by ALK > Active ALK phosphorylates PLCG1',
 'Signaling by ALK > Active ALK phosphorylates SHC1',
 'Signaling by ALK > Active ALK phosphorylates SRC',
 'Signaling by ALK > Active ALK recruits PI3K',
 'Signaling by ALK > MDK and PTN in ALK signaling > (PTN,MDK):PTPRZ oligomerizes',
 'Signaling by ALK > MDK and PTN in ALK signaling > ACTIVATION',
 'Signaling by ALK > M